In [1]:
import os
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# 0. Setup - unduh resource NLTK yang diperlukan (hanya perlu sekali)
for pkg in ["stopwords", "wordnet", "omw-1.4", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

STOPWORDS = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

# Konfigurasi folder input/output -> ubah sesuai lokasi file di komputer Anda
INPUT_DIR = "."          # folder tempat 5 file csv berada
OUTPUT_DIR = "./processed"  # folder hasil pemrosesan akan disimpan
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. FUNGSI CLEANING (dipakai semua kolom teks)
def basic_clean(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)                # hapus HTML tag
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # hapus URL
    text = re.sub(r"[\r\n\t]+", " ", text)               # hapus newline/tab
    text = re.sub(r"[^A-Za-z0-9\s.,!?'-]", " ", text)    # hapus karakter aneh
    text = re.sub(r"\s+", " ", text).strip()             # rapikan spasi
    return text


def light_clean(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

# 2-5. PIPELINE LENGKAP: case folding -> tokenization -> stopword -> lemma
def full_preprocess(text: str) -> str:
    text = basic_clean(text)
    text = text.lower()                                   # 2. case folding
    tokens = word_tokenize(text)                           # 3. tokenization
    tokens = [t for t in tokens if t.isalpha()]             # buang angka & tanda baca
    tokens = [t for t in tokens if t not in STOPWORDS]      # 4. stopword removal
    tokens = [lemmatizer.lemmatize(t) for t in tokens]      # 5. lemmatization
    return " ".join(tokens)

# PROSES SETIAP FILE CSV
def process_movies():
    print("Memproses movies.csv ...")
    df = pd.read_csv(f"{INPUT_DIR}/movies.csv")
    df["title_clean"] = df["title"].apply(light_clean)
    df["overview_clean"] = df["overview"].apply(basic_clean)
    df["overview_processed"] = df["overview"].apply(full_preprocess)
    df.to_csv(f"{OUTPUT_DIR}/movies_processed.csv", index=False)
    return df


def process_reviews():
    print("Memproses reviews.csv ...")
    df = pd.read_csv(f"{INPUT_DIR}/reviews.csv")
    df["author_clean"] = df["author"].apply(light_clean)
    df["content_clean"] = df["content"].apply(basic_clean)
    df["content_processed"] = df["content"].apply(full_preprocess)
    df.to_csv(f"{OUTPUT_DIR}/reviews_processed.csv", index=False)
    return df


def process_cast():
    print("Memproses cast.csv ...")
    df = pd.read_csv(f"{INPUT_DIR}/cast.csv")
    df["actor_clean"] = df["actor"].apply(light_clean)
    df["character_clean"] = df["character"].apply(light_clean)
    df.to_csv(f"{OUTPUT_DIR}/cast_processed.csv", index=False)
    return df


def process_crew():
    print("Memproses crew.csv ...")
    df = pd.read_csv(f"{INPUT_DIR}/crew.csv")
    df["name_clean"] = df["name"].apply(light_clean)
    df["job_clean"] = df["job"].apply(light_clean)
    df.to_csv(f"{OUTPUT_DIR}/crew_processed.csv", index=False)
    return df


def process_genres():
    print("Memproses genres.csv ...")
    df = pd.read_csv(f"{INPUT_DIR}/genres.csv")
    df["genre_clean"] = df["genre"].apply(light_clean)
    df.to_csv(f"{OUTPUT_DIR}/genres_processed.csv", index=False)
    return df


if __name__ == "__main__":
    process_movies()
    process_reviews()
    process_cast()
    process_crew()
    process_genres()
    print(f"\nSelesai! Semua file hasil pemrosesan tersimpan di folder: {OUTPUT_DIR}/")

Memproses movies.csv ...
Memproses reviews.csv ...
Memproses cast.csv ...
Memproses crew.csv ...
Memproses genres.csv ...

Selesai! Semua file hasil pemrosesan tersimpan di folder: ./processed/
